# Deep Past Initiative – Machine Translation (Inference Notebook)

This notebook is a **starter / baseline** for this Kaggle competition.

Training Code is [here](https://www.kaggle.com/code/takamichitoda/dpc-starter-train).

# A Rule-Based Baseline Solution (TR-TRY Notebook)

This notebook builds a **"translation memory bank"** and **"bidirectional confidence dictionary"** using the training set, and generates translation results by matching the most similar training samples for test set texts through multi-dimensional retrieval.

TR-TRY Notebook is [here](https://www.kaggle.com/code/jackcerion/tr-try). **(copy from @jackcerion)**

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

In [ ]:
#MODEL_PATH = "/kaggle/input/dpc-starter-train/byt5-akkadian-model/"
#MODEL_PATH = "/kaggle/input/epoch30-seed42/byt5-akkadian-model"
# MODEL_PATH="/kaggle/input/k/qifeihhh666/dpc-starter-train/byt5-akkadian-model/"
# MODEL_PATH="/kaggle/input/k/shwesh/dpc-starter-train/arab-engl-akkadian-model/"
# MODEL_PATH="/kaggle/input/notebooks/shwesh/dpc-starter-train/byt5-akkadian-model/"
# MODEL_PATH="/kaggle/input/notebooks/shwesh/dpc-backtranslation-augmented-train/byt5-akkadian-model/"
# MODEL_PATH="/kaggle/input/notebooks/shwesh/dpc-generic-train/byt5-akkadian-model/"
MODEL_PATH="/kaggle/input/notebooks/shwesh/post-comp-baseline/byt5-akkadian-model/"

In [ ]:
TEST_DATA_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
BATCH_SIZE = 16
MAX_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Model Loading ---
print(f"Loading model from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)
model.eval()

# --- Data Preparation ---
test_df = pd.read_csv(TEST_DATA_PATH)

In [ ]:
PREFIX = ""#I TRAINED WITHOUT WHOOPS "translate Akkadian to English: "

class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['transliteration'].astype(str).tolist()
        self.texts = [PREFIX + i for i in self.texts]
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text, 
            max_length=MAX_LENGTH, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }

test_dataset = InferenceDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- Inference Loop ---
print("Starting Inference...")
all_predictions = []

In [ ]:
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
  
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            num_beams=4,
            early_stopping=True
        )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend([d.strip() for d in decoded])

In [ ]:
# --- Submission ---
submission_Deep = pd.DataFrame({
    "id": test_df["id"],
    "translation": all_predictions
})

submission_Deep["translation"] = submission_Deep["translation"].apply(lambda x: x if len(x) > 0 else "broken text")

submission_Deep.to_csv("submission.csv", index=False)
submission_Deep.head()

In [ ]:
#TODO OUTPUT 10 OR SO FROM 2nd places EVAL DATASET WITH TRUTH VALUES

In [ ]:
EVAL_DATA_PATH = "/kaggle/input/datasets/wukeneth/akkadian-v9-experi-dataset/eval.json"
# BATCH_SIZE = 16
# MAX_LENGTH = 512
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# # --- Model Loading ---
# print(f"Loading model from {MODEL_PATH}...")
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)
# model.eval()

# --- Data Preparation ---
eval_df = pd.read_json(EVAL_DATA_PATH)

In [ ]:
class InferenceEvalDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['source'].astype(str).tolist()
        self.texts = [PREFIX + i for i in self.texts]
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text, 
            max_length=MAX_LENGTH, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }

eval_dataset = InferenceEvalDataset(eval_df, tokenizer)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- Inference Loop ---
print("Starting Inference...")
all_predictions = []

In [ ]:
with torch.no_grad():
    for batch in tqdm(eval_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
  
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            num_beams=4,
            early_stopping=True
        )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend([d.strip() for d in decoded])

In [ ]:
# --- Evaluation ---
evaluation_Deep = pd.DataFrame({
    "target": eval_df["target"],
    "translation": all_predictions,
})

evaluation_Deep["translation"] = evaluation_Deep["translation"].apply(lambda x: x if len(x) > 0 else "broken text")

evaluation_Deep.to_csv("evaluation.csv", index=False)
evaluation_Deep.head()

In [ ]:
# RUN_EVALUATION
from sacrebleu.metrics import BLEU, CHRF

refs = evaluation_Deep["target"].tolist()
final_preds = evaluation_Deep["translation"].tolist()

bleu = BLEU()
chrf = CHRF()
b_score = bleu.corpus_score(final_preds, [refs]).score
c_score = chrf.corpus_score(final_preds, [refs]).score
combo = (b_score * c_score) ** 0.5

print(f"BLEU:  {b_score:.2f}")
print(f"chrF:  {c_score:.2f}")
print(f"Combo: {combo:.2f}")

In [ ]:
def perform_mbr(dfs):
    candidate_cols = [col for col in dfs[0].columns if col != 'id']
    num_rows = len(dfs[0])
    
    mbr_results = []

    for i in tqdm(range(num_rows), desc="Processing MBR"):
        row_id = dfs[0].loc[i, 'id']

        candidates = []
        for df in dfs:
            for col in candidate_cols:
                val = df.loc[i, col]
                candidates.append(str(val).strip() if pd.notna(val) else "")
                
        unique_candidates = list(set([c for c in candidates if c]))
        
        best_score = -1.0
        best_candidate = ""

        n_unique = len(unique_candidates)
        if n_unique == 0:
            mbr_results.append({'id': row_id, 'translation': ""})
            continue
        if n_unique == 1:
            mbr_results.append({'id': row_id, 'translation': unique_candidates[0]})
            continue
        
        for hyp in unique_candidates:
            total_score = 0.0

            for ref in unique_candidates:
                if hyp != ref: 
                    total_score += sentence_level_score(hyp, ref)

            expected_score = total_score / max(1, n_unique - 1)

            if expected_score > best_score:
                best_score = expected_score
                best_candidate = hyp

        mbr_results.append({
            'id': row_id,
            'translation': best_candidate
        })
        
    return pd.DataFrame(mbr_results)

In [ ]:
#Todo: 
## Make a generic function to run the model.
## Run the model 3 times on the submission data
## Append each output to a dataframe, then feed that dataframe into MBR. And export as submission.csv
## Then, repeat running the model using the eval data.